In [0]:
%run ./00_setup

In [0]:
random.seed(42)

country_currency = {
    "ES": "EUR",
    "MX": "MXN",
    "AR": "ARS",
    "CO": "COP",
    "CL": "CLP"
}

segments = ["B2C", "SMB", "ENT"]

def rand_date(start: dt.date, end: dt.date) -> dt.date:
    delta = (end - start).days
    return start + dt.timedelta(days=random.randint(0, delta))

customers = []

for i in range(1, 501):
    customer_id = f"C{i:05d}"
    country = random.choice(list(country_currency.keys()))
    signup_date = rand_date(dt.date(2022, 1, 1), dt.date(2025, 1, 31))
    segment = random.choice(segments)
    email = f"customer{i}@demo.com"
    customers.append((customer_id, signup_date, country, segment, email))

# Errores intencionales
customers.append(("C00010", dt.date(2024, 1, 10), "ES", "B2C", "duplicated_customer@demo.com"))  # duplicado
customers.append(("C00999", dt.date(2030, 1, 1), "MX", "SMB", "future@demo.com"))                 # fecha futura
customers.append(("C01000", dt.date(2024, 5, 10), "AR", "ENT", "bad-email-format"))               # email inválido

customers_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("segment", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
])

customers_df = spark.createDataFrame(customers, schema=customers_schema)

display(customers_df.limit(10))
print(customers_df.count())

In [0]:
payment_methods = ["CARD", "TRANSFER", "CASH", "PAYPAL"]
order_statuses = ["CREATED", "PAID", "CANCELLED", "REFUNDED"]

customer_ids = [f"C{i:05d}" for i in range(1, 501)]
customer_country_map = {row["customer_id"]: row["country"] for row in customers_df.select("customer_id", "country").collect() if row["customer_id"] in customer_ids}

orders = []

for i in range(1, 5001):
    order_id = f"O{i:06d}"
    customer_id = random.choice(customer_ids)
    country = customer_country_map[customer_id]
    currency = country_currency[country]
    quantity = random.randint(1, 5)
    unit_price = round(random.uniform(5, 500), 2)
    order_total = round(quantity * unit_price, 2)
    order_ts = dt.datetime(
        2024,
        random.randint(1, 12),
        random.randint(1, 28),
        random.randint(0, 23),
        random.randint(0, 59),
        0
    )
    payment_method = random.choice(payment_methods)
    order_status = random.choice(order_statuses)
    email = f"{customer_id.lower()}@orders.demo.com"
    ingest_date = dt.date.today()

    orders.append([
        order_id, customer_id, order_ts, country, currency,
        quantity, unit_price, order_total,
        payment_method, order_status, email, ingest_date
    ])

# Errores intencionales
orders[10][0] = None                           # order_id nulo
orders[20][0] = orders[21][0]                  # duplicado
orders[30][1] = None                           # customer_id nulo
orders[40][5] = -3                             # quantity negativa
orders[50][4] = "USD"                          # moneda inválida
orders[60][3] = "ZZ"                           # país inválido
orders[70][10] = "correo-sin-formato"          # email inválido
orders[80][2] = dt.datetime(2035, 1, 1, 10, 0, 0)  # fecha futura
orders[90][7] = 999999.99                      # total anómalo
orders[100][7] = round((orders[100][5] * orders[100][6]) + 10, 2)  # total inconsistente
orders[110][1] = "C99999"                      # FK inválida

orders_schema = T.StructType([
    T.StructField("order_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("currency", T.StringType(), True),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("unit_price", T.DoubleType(), True),
    T.StructField("order_total", T.DoubleType(), True),
    T.StructField("payment_method", T.StringType(), True),
    T.StructField("order_status", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("ingest_date", T.DateType(), True),
])

orders_df = spark.createDataFrame(orders, schema=orders_schema)

display(orders_df.limit(10))

In [0]:
customers_df.write.mode("overwrite").format("delta").saveAsTable(f"workspace.bronze.customers")

In [0]:
orders_df.write.mode("overwrite").format("delta").saveAsTable(f"workspace.bronze.orders")